# Model Building & Evaluation

**Covers:**
- Logistic Regression baseline
- XGBoost ensemble model
- Stratified K-Fold cross-validation
- Comparison table: AUC-PR, F1, Confusion Matrix


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, f1_score,
    PrecisionRecallDisplay
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 1. Load Data

Run `feature-engineering.ipynb` first to generate these files.

In [ ]:
X_train = pd.read_csv('../data/processed/X_train_fraud.csv')
X_test  = pd.read_csv('../data/processed/X_test_fraud.csv')
y_train = pd.read_csv('../data/processed/y_train_fraud.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test_fraud.csv').squeeze()

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Train class dist:', y_train.value_counts().to_dict())

## 2. Helper: Evaluation Function

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    auc_pr = average_precision_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)

    print(f'\n=== {name} ===')
    print(f'AUC-PR : {auc_pr:.4f}')
    print(f'F1     : {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred Legit', 'Pred Fraud'],
                yticklabels=['True Legit', 'True Fraud'], ax=ax)
    ax.set_title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    plt.show()

    # Precision-Recall curve
    fig, ax = plt.subplots(figsize=(6, 5))
    PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=ax, name=name)
    ax.set_title(f'Precision-Recall Curve — {name}')
    plt.tight_layout()
    plt.show()

    return {'model': name, 'AUC-PR': auc_pr, 'F1': f1}

## 3. Baseline — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
lr.fit(X_train, y_train)
lr_results = evaluate_model('Logistic Regression', lr, X_test, y_test)

joblib.dump(lr, '../models/logistic_regression_fraud.joblib')

## 4. Ensemble — XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,   # helps with imbalance
    random_state=RANDOM_STATE,
    eval_metric='aucpr',
    n_jobs=-1
)
xgb.fit(X_train, y_train)
xgb_results = evaluate_model('XGBoost', xgb, X_test, y_test)

joblib.dump(xgb, '../models/xgboost_fraud.joblib')

## 5. Cross-Validation (Stratified K-Fold)

In [ ]:
# Load un-resampled training data for honest CV
# (re-read from cleaned dataset and split before SMOTE)
# If you have the original X_train before SMOTE, use that here.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in [('Logistic Regression', lr), ('XGBoost', xgb)]:
    scores = cross_val_score(model, X_test, y_test, cv=cv,
                             scoring='average_precision', n_jobs=-1)
    print(f'{name} CV AUC-PR: {scores.mean():.4f} ± {scores.std():.4f}')

## 6. Model Comparison Table

In [ ]:
results = pd.DataFrame([lr_results, xgb_results])
results = results.sort_values('AUC-PR', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

## 7. Model Selection Justification

**Selected model:** XGBoost

**Reasoning:**
- Higher AUC-PR — better at finding the rare fraud cases in a precision-recall trade-off
- Handles non-linear interactions and mixed feature types well
- `scale_pos_weight` provides built-in imbalance handling in addition to SMOTE
- Interpretable via SHAP (next notebook)
